# Pair cells with H&E

The point of sMCC40M is that each expression profile has **its own** image patch. This tutorial
reads native-resolution crops around cell centroids with the same reader the benchmark uses,
then builds a small image–expression table.

> **Code-only page.** Outputs appear when you run the notebook on the downloaded sample.

> **Requirements.** `pip install tifffile zarr pillow anndata pyarrow pandas matplotlib` (OpenSlide is optional) · the sMMC40M code repository cloned next to your data · HD-002 downloaded.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import anndata as ad
import tifffile
import matplotlib.pyplot as plt

REPO = Path("sMMC40M")                       # git clone https://github.com/aicb-ZhangLabs/sMMC40M.git
sys.path.insert(0, str(REPO / "benchmark"))
from image_reader import RGBReader           # same crop logic as the benchmark

SAMPLE_ROOT = Path("smcc_data/HD/samples/HD-002")
sample = pq.read_table(SAMPLE_ROOT / "metadata/samples.parquet").to_pylist()[0]
print(sample["sample_id"], sample["stain"])

## 1. Open the image

`RGBReader` reads only the requested window (OpenSlide if installed, otherwise a memory-mapped
TIFF). It needs the image shape, which we take from the file itself.

In [ ]:
image_path = SAMPLE_ROOT / sample["image_file"]
with tifffile.TiffFile(image_path) as tf:
    series = tf.series[0]
    shape_yx = (series.shape[series.axes.index("Y")], series.shape[series.axes.index("X")])
reader = RGBReader(image_path, shape_yx)
print("image", shape_yx, "via", getattr(reader, "backend", "openslide"))

## 2. Choose cells

Use candidates that have an expression row. The release also ships a per-candidate
**eligibility** table (`eligibility_file`) with technical flags such as whether the centroid and
mask fall in the selected image; inspect it before large-scale training.

In [ ]:
cells = pq.read_table(SAMPLE_ROOT / "metadata/cells.parquet").to_pandas()
cells = cells.dropna(subset=["h5ad_row"]).copy()
cells["h5ad_row"] = cells["h5ad_row"].astype(int)

elig_path = SAMPLE_ROOT / sample["eligibility_file"] if sample.get("eligibility_file") else None
if elig_path is not None and elig_path.exists():
    elig = pq.read_table(elig_path).to_pandas()
    print("eligibility columns:", elig.columns.tolist())

# keep cells whose full crop lies inside the image
SIZE = 150                                   # benchmark crop: 150 × 150 native pixels
h, w = shape_yx
inside = (cells.x_px >= SIZE / 2) & (cells.x_px < w - SIZE / 2) & \
         (cells.y_px >= SIZE / 2) & (cells.y_px < h - SIZE / 2)
cells = cells[inside]
print(f"{len(cells):,} cells with a complete {SIZE}px crop")

## 3. Look at the cells

Crop sizes are in **native pixels**; the physical window depends on the image's microns per pixel.

In [ ]:
mpp = sample.get("mpp_x_um_per_px")
if mpp:
    print(f"{SIZE} px ≈ {SIZE * mpp:.0f} µm at {mpp:.3f} µm/px")

picked = cells.sample(16, random_state=0)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for ax, (_, c) in zip(axes.flat, picked.iterrows()):
    ax.imshow(reader.crop(c.x_px, c.y_px, size=SIZE))
    ax.set_title(str(c.cell_id)[-10:], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Image and expression side by side

Pick a gene and compare crops from high- and low-expressing cells.

In [ ]:
adata = ad.read_h5ad(SAMPLE_ROOT / sample["expression_file"], backed="r")
genes = pq.read_table(SAMPLE_ROOT / sample["genes_file"]).to_pandas()
symbol_col = next((c for c in genes.columns if "symbol" in c.lower()), "gene_id")

GENE = genes.loc[genes["feature_type"] == "Gene Expression", symbol_col].iloc[0]   # replace with a gene you care about
g = int(np.flatnonzero(genes[symbol_col].astype(str).values == str(GENE))[0])

rows = np.sort(cells["h5ad_row"].sample(3000, random_state=0).values)
expr = np.asarray(adata.X[rows][:, g].todense()).ravel()
sub = cells.set_index("h5ad_row").loc[rows].assign(expr=expr)
high, low = sub.nlargest(8, "expr"), sub.nsmallest(8, "expr")

fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, (_, c) in zip(axes[0], high.iterrows()):
    ax.imshow(reader.crop(c.x_px, c.y_px, SIZE)); ax.set_title(f"{GENE}={int(c.expr)}", fontsize=8); ax.axis("off")
for ax, (_, c) in zip(axes[1], low.iterrows()):
    ax.imshow(reader.crop(c.x_px, c.y_px, SIZE)); ax.set_title(f"{GENE}={int(c.expr)}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. A training table

A minimal image→expression dataset: crops as arrays, `log1p` raw counts as targets (the benchmark
target), and IDs for joins. Swap the crops for features from any pathology encoder.

In [ ]:
batch = cells.sample(256, random_state=1).sort_values("h5ad_row")
X_img = np.stack([np.asarray(reader.crop(c.x_px, c.y_px, SIZE)) for _, c in batch.iterrows()])
gene_mask = (genes["feature_type"] == "Gene Expression").values
Y = np.log1p(np.asarray(adata.X[batch["h5ad_row"].values].todense())[:, gene_mask])
print("images", X_img.shape, "targets", Y.shape)

In [ ]:
adata.file.close(); reader.close()

> **Reproducing the benchmark inputs.** The benchmark encodes these crops with CONCH (`smcc_benchmark.py prepare` / `encode`) and fits a PCA + Ridge readout. [Tutorial 5](05_benchmark.ipynb) runs it.